# COJO joint support of additional credible sets: SuSiE-RSS vs SuSiE-rRSS (data processing)

Joint COJO models from notebook `24` in the study × region pairs with multiple credible sets, for each method:

- **SuSiE-rRSS:** `[cojo]` job `cojo_multi_cs` (56 pairs).
- **SuSiE-RSS:** `[cojo]` job `cojo_multi_cs_susie_rss` (82 pairs).

In each pair, every credible set enters the joint model through its top-PIP variant. The **lead** set is the one whose top variant has the largest marginal |z|; every other set is **additional** (2nd, 3rd, ...). Additional sets fall in three groups:

- **SuSiE-rRSS:** additional sets of SuSiE-rRSS.
- **SuSiE-RSS, shared:** SuSiE-RSS sets sharing a variant with a SuSiE-rRSS credible set in the same study × region.
- **SuSiE-RSS only:** SuSiE-RSS sets sharing no variant with any SuSiE-rRSS credible set.

Each summary is shown for all loci and excluding *APOE* (block `chr19_42346101_46842901`). SuSiE-rRSS reports one credible set at *APOE* in every dataset (SER fallback), so *APOE* enters only the SuSiE-RSS run.

COJO uses the same ADSP reference LD as fine-mapping, so joint support is not independent validation; a set that loses joint support even under that LD is the stronger evidence.

**Outputs:** `figure_data/COJO.SuSiE_RSS_vs_rRSS_Joint` (one row per credible set), `figure_data/COJO.SuSiE_RSS_vs_rRSS_Summary` (additional sets per locus set and group), `figure_data/COJO.SuSiE_RSS_vs_rRSS_SubGWS` (the Fig 6f,g selection for both methods; plotted by `Figures/Figure_6/COJO_SuSiE_RSS_vs_rRSS`)

In [7]:
source("../../R/utils.R")

fm_root    <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10"
cojo_dir   <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
runs       <- c("SuSiE-rRSS" = "cojo_multi_cs", "SuSiE-RSS" = "cojo_multi_cs_susie_rss")
apoe_block <- "chr19_42346101_46842901"

## -log10 two-sided P from z, without underflow for very large |z|
mlog10p <- function(z) -(log(2) + pnorm(-abs(z), log.p = TRUE)) / log(10)
cs_rows <- function(df) filter(df, !is.na(cs_coverage_0.95), !cs_coverage_0.95 %in% c("", "0", "susie_rss_0"))

## Joint models of both runs

In [8]:
## one row per CS: its top-PIP variant in the joint model, with the marginal beta from <unit>.ma
read_joint <- function(job, method)
  list.files(file.path(cojo_dir, job, "result"), pattern = "\\.cojo_joint\\.tsv$", recursive = TRUE, full.names = TRUE) %>%
    map_dfr(function(f) {
      ma <- read_tsv(sub("\\.cojo_joint\\.tsv$", ".ma", f), show_col_types = FALSE) %>% select(SNP, marginal_beta = b)
      read_tsv(f, show_col_types = FALSE) %>% left_join(ma, by = "SNP")
    }) %>%
    mutate(method = method)

## SuSiE-RSS sets sharing a variant with a SuSiE-rRSS set in the same study x region
rrss_var <- read_gwas_tables(file.path(fm_root, "AD_GWAS_rss_ebmix_r4_L10/export/")) %>% cs_rows() %>%
  distinct(study, block_id, variant_ID) %>% mutate(hit = TRUE)
rss_in_rrss <- read_gwas_tables(file.path(fm_root, "AD_GWAS_rss_baseline_r4_L10/export/")) %>% cs_rows() %>%
  transmute(study, block_id, cs = cs_coverage_0.95, variant_ID) %>%
  left_join(rrss_var, by = c("study", "block_id", "variant_ID")) %>%
  group_by(study, block_id, cs) %>% summarise(in_rrss = any(hit %in% TRUE), .groups = "drop") %>%
  mutate(method = "SuSiE-RSS")

ser_flag <- read_tsv(file.path(fm_root, "AD_GWAS_rss_ebmix_r4_L10/export/SER_flag.tsv"), show_col_types = FALSE) %>%
  transmute(study, block_id, ser_fallback = serFallbackUsed %in% TRUE)

joint <- imap_dfr(runs, read_joint) %>%
  group_by(method, study, block_id) %>%
  mutate(n_cs_pair = n(),
         role = if_else(row_number(desc(abs(marginal_z))) == 1, "lead", "additional")) %>%
  ungroup() %>%
  left_join(rss_in_rrss, by = c("method", "study", "block_id", "cs")) %>%
  left_join(ser_flag, by = c("study", "block_id")) %>%
  mutate(group = case_when(method == "SuSiE-rRSS" ~ "SuSiE-rRSS",
                           in_rrss %in% TRUE    ~ "SuSiE-RSS, shared",
                           TRUE                 ~ "SuSiE-RSS only"),
         group            = factor(group, levels = c("SuSiE-rRSS", "SuSiE-RSS, shared", "SuSiE-RSS only")),
         locus            = if_else(block_id == apoe_block, "APOE", "non-APOE"),
         study_label      = unname(cohort_labels[study]),
         ser_fallback     = ser_fallback %in% TRUE,
         marginal_mlog10p = mlog10p(marginal_z),
         joint_mlog10p    = mlog10p(joint_z),
         beta_ratio       = abs(joint_beta) / abs(marginal_beta),
         sign_flip        = sign(joint_beta) != sign(marginal_beta))

## coverage: pairs and CSs per method, joint fits missing, and the APOE share
joint %>% group_by(method) %>%
  summarise(n_pairs = n_distinct(paste(study, block_id)), n_cs = n(), n_joint_missing = sum(is.na(joint_z)),
            n_apoe_pairs = n_distinct(study[locus == "APOE"]), n_apoe_cs = sum(locus == "APOE"), .groups = "drop")

method,n_pairs,n_cs,n_joint_missing,n_apoe_pairs,n_apoe_cs
<chr>,<int>,<int>,<int>,<int>,<int>
SuSiE-RSS,82,235,0,10,64
SuSiE-rRSS,56,121,0,0,0


## Additional credible sets: all loci and excluding *APOE*

`joint_p_gws` / `joint_p_1e5`: joint P < 5 × 10⁻⁸ / < 10⁻⁵. `stronger_jointly`: joint P smaller than marginal P. `beta_ratio`: |β_joint| / |β_marginal|. `sign_flip` and `ratio_gt2` flag joint effects inflated by collinear representatives (artefacts, not support).

In [9]:
locus_sets <- function(df) bind_rows(
  mutate(df, locus_set = "All loci"),
  filter(df, locus == "non-APOE") %>% mutate(locus_set = "Excluding APOE")) %>%
  mutate(locus_set = factor(locus_set, levels = c("All loci", "Excluding APOE")))

summ <- joint %>% filter(role == "additional", !is.na(joint_z)) %>% locus_sets() %>%
  group_by(locus_set, group, .drop = FALSE) %>%
  summarise(n_pairs           = n_distinct(paste(study, block_id)),
            n_cs              = n(),
            joint_p_gws       = sum(joint_mlog10p > -log10(5e-8)),
            joint_p_1e5       = sum(joint_mlog10p > -log10(1e-5)),
            no_joint_support  = sum(joint_mlog10p <= -log10(1e-5)),
            marginal_p_gws    = sum(marginal_mlog10p > -log10(5e-8)),
            stronger_jointly  = sum(joint_mlog10p > marginal_mlog10p),
            median_beta_ratio = round(median(beta_ratio), 2),
            sign_flip         = sum(sign_flip),
            ratio_gt2         = sum(beta_ratio > 2),
            .groups = "drop") %>%
  mutate(pct_no_joint_support = round(100 * no_joint_support / n_cs, 1),
         pct_stronger_jointly = round(100 * stronger_jointly / n_cs, 1))
as.data.frame(summ)

locus_set,group,n_pairs,n_cs,joint_p_gws,joint_p_1e5,no_joint_support,marginal_p_gws,stronger_jointly,median_beta_ratio,sign_flip,ratio_gt2,pct_no_joint_support,pct_stronger_jointly
<fct>,<fct>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<dbl>,<dbl>
All loci,SuSiE-rRSS,56,65,55,65,0,40,52,1.05,0,2,0.0,80.0
All loci,"SuSiE-RSS, shared",56,64,54,64,0,39,51,1.06,0,2,0.0,79.7
All loci,SuSiE-RSS only,31,89,58,65,24,70,22,0.81,30,16,27.0,24.7
Excluding APOE,SuSiE-rRSS,56,65,55,65,0,40,52,1.05,0,2,0.0,80.0
Excluding APOE,"SuSiE-RSS, shared",56,64,54,64,0,39,51,1.06,0,2,0.0,79.7
Excluding APOE,SuSiE-RSS only,21,35,9,16,19,20,9,0.87,13,4,54.3,25.7


## Where the SuSiE-RSS-only sets are

In [10]:
rss_only <- joint %>% filter(role == "additional", group == "SuSiE-RSS only")

## SuSiE-RSS-only sets in regions where SuSiE-rRSS used the SER fallback
rss_only %>% count(locus, ser_fallback, name = "n_cs")

## blocks contributing the most SuSiE-RSS-only sets
rss_only %>% group_by(block_id) %>%
  summarise(n_cs = n(), n_studies = n_distinct(study),
            no_joint_support = sum(joint_mlog10p <= -log10(1e-5), na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(n_cs)) %>% head(15)

locus,ser_fallback,n_cs
<chr>,<lgl>,<int>
APOE,FALSE,8
APOE,TRUE,46
non-APOE,FALSE,22
non-APOE,TRUE,13


block_id,n_cs,n_studies,no_joint_support
<chr>,<int>,<int>,<int>
chr19_42346101_46842901,54,10,5
chr11_56858541_63818332,6,1,6
chr17_45383525_50162864,6,1,4
chr11_84267999_86714492,3,3,0
chr1_205972031_208461272,3,2,3
chr20_55439357_57610823,3,1,3
chr2_125689597_127728648,3,2,2
chr19_90949_1507536,2,2,0
chr8_26225312_29469590,2,2,0


## Fig 6f,g selection for both methods

As Fig 6f,g (`Data_Processing/Figure_6/Fig6FG_S12B`): every credible-set representative (lead and additional) in the multi-CS pairs whose marginal P > 5 × 10⁻⁸, with its joint P. SuSiE-rRSS on all loci should give the 26 representatives of Fig 6f,g.

In [11]:
sub_gws <- joint %>%
  filter(!is.na(joint_z), marginal_mlog10p < -log10(5e-8)) %>%
  locus_sets() %>%
  mutate(method = factor(method, levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         change = factor(if_else(joint_mlog10p > marginal_mlog10p, "Stronger jointly", "Weaker jointly"),
                         levels = c("Stronger jointly", "Weaker jointly")))

sub_gws %>% group_by(locus_set, method) %>%
  summarise(n = n(), n_pairs = n_distinct(paste(study, block_id)),
            stronger_jointly = sum(change == "Stronger jointly"),
            joint_p_gws = sum(joint_mlog10p > -log10(5e-8)), joint_p_1e5 = sum(joint_mlog10p > 5),
            .groups = "drop")
## SuSiE-rRSS, all loci: the 26 representatives of Fig 6f,g
sum(sub_gws$method == "SuSiE-rRSS" & sub_gws$locus_set == "All loci")

locus_set,method,n,n_pairs,stronger_jointly,joint_p_gws,joint_p_1e5
<fct>,<fct>,<int>,<int>,<int>,<int>,<int>
All loci,SuSiE-RSS,45,39,36,23,36
All loci,SuSiE-rRSS,26,25,25,16,26
Excluding APOE,SuSiE-RSS,41,36,32,19,32
Excluding APOE,SuSiE-rRSS,26,25,25,16,26


[1] 26

In [12]:
save_data(joint, "COJO.SuSiE_RSS_vs_rRSS_Joint")
save_data(summ,  "COJO.SuSiE_RSS_vs_rRSS_Summary")
save_data(sub_gws, "COJO.SuSiE_RSS_vs_rRSS_SubGWS")